In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [38]:
import os
from os.path import exists
import sys
sys.path.append('..')

In [3]:
import random
import numpy as np
import pylab as plt

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, SubprocVecEnv
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [45]:
from vimms.Common import POSITIVE, set_log_level_warning, load_obj, save_obj
from vimms.ChemicalSamplers import MZMLFormulaSampler, MZMLRTandIntensitySampler, MZMLChromatogramSampler
from vimms.Noise import UniformSpikeNoise
from vimms.Evaluation import evaluate_real
from vimms.Chemicals import ChemicalMixtureFromMZML
from vimms.Roi import RoiBuilderParams, SmartRoiParams

from mass_spec_utils.data_import.mzmine import load_picked_boxes

from vimms_gym.env import DDAEnv
from vimms_gym.chemicals import generate_chemicals
from vimms_gym.features import obs_to_dfs
from vimms_gym.evaluation import evaluate
from vimms_gym.policy import fullscan_policy, random_policy, topN_policy, best_ppo_policy
from vimms_gym.evaluation import evaluate, run_method

# 1. Parameters

In [6]:
n_chemicals = (5000, 20000)
mz_range = (70, 1000)
rt_range = (0, 1440)
intensity_range = (1E4, 1E20)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 15
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1e3

In [9]:
mzml_filename = 'fullscan_QCB.mzML'    
samplers = None
samplers_pickle = 'samplers_%s.p' % mzml_filename
if exists(samplers_pickle):
    logger.info('Loaded %s' % samplers_pickle)
    samplers = load_obj(samplers_pickle)
    mz_sampler = samplers['mz']
    ri_sampler = samplers['rt_intensity']
    cr_sampler = samplers['chromatogram']
else:    
    logger.info('Creating samplers from %s' % mzml_filename)
    mz_sampler = MZMLFormulaSampler(mzml_filename, min_mz=min_mz, max_mz=max_mz)
    ri_sampler = MZMLRTandIntensitySampler(mzml_filename, min_rt=min_rt, max_rt=max_rt,
                                           min_log_intensity=min_log_intensity,
                                           max_log_intensity=max_log_intensity)
    cr_sampler = MZMLChromatogramSampler(mzml_filename)
    samplers = {
        'mz': mz_sampler,
        'rt_intensity': ri_sampler,
        'chromatogram': cr_sampler
    }
    save_obj(samplers, samplers_pickle)

2022-03-31 20:17:56.573 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 2471 scans
2022-03-31 20:17:59.508 | DEBUG    | mass_spec_utils.data_import.mzml:_load_file:166 - Loaded 2471 scans
2022-03-31 20:21:08.155 | DEBUG    | vimms.ChemicalSamplers:_extract_rois:491 - Extracted 516978 good ROIs from fullscan_QCB.mzML


In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [11]:
max_peaks = 100
in_dir = 'results_%d_%d' % (max_peaks, rt_tol)

In [12]:
n_eval_episodes = 10
deterministic = True

# 2. Evaluation

#### Generate some chemical sets

In [13]:
n_eval_episodes = 1
eval_dir = 'evaluation'
methods = [
    'PPO',
    'TopN',
    'random',    
]

In [14]:
chemical_creator_params = params['chemical_creator']

chem_list = []
for i in range(n_eval_episodes):
    print(i)
    chems = generate_chemicals(chemical_creator_params)
    chem_list.append(chems)

0


2022-03-31 20:21:10.629 | DEBUG    | vimms.Chemicals:sample:468 - Sampled rt and intensity values and chromatograms


#### Run different methods

In [15]:
len(chem_list[0])

2436

In [16]:
max_peaks

100

In [17]:
out_dir = '%s/eval_%d_%d' % (eval_dir, max_peaks, rt_tol)
in_dir, out_dir

('results_100_15', 'evaluation/eval_100_15')

#### Compare to Top-10

In [18]:
copy_params = dict(params)
copy_params['env']['rt_tol'] = rt_tol

for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, N=10, print_eval=True)
    print()

method = PPO max_peaks = 100 rt_tol = 15

Episode 0 finished after 3812 timesteps with reward 596.2293125686161
{'coverage_prop': '0.614', 'intensity_prop': '0.437', 'ms1/ms2 ratio': '0.052', 'efficiency': '0.413'}

method = TopN max_peaks = 100 rt_tol = 15

Episode 0 finished after 3656 timesteps with reward 818.7776813269912
{'coverage_prop': '0.721', 'intensity_prop': '0.625', 'ms1/ms2 ratio': '0.104', 'efficiency': '0.530'}

method = random max_peaks = 100 rt_tol = 15

Episode 0 finished after 3964 timesteps with reward 776.5770110369806
{'coverage_prop': '0.740', 'intensity_prop': '0.576', 'ms1/ms2 ratio': '0.009', 'efficiency': '0.459'}



#### Test classic Top-N controller in ViMMS

In [19]:
from vimms.MassSpec import IndependentMassSpectrometer
from vimms.Controller import TopNController
from vimms.Environment import Environment

In [20]:
chemicals = chem_list[0]
len(chemicals)

2436

In [21]:
set_log_level_warning()

7

In [22]:
noise_params = params['noise']
noise_density = noise_params['noise_density']
noise_max_val = noise_params['noise_max_val']
noise_min_mz = noise_params['mz_range'][0]
noise_max_mz = noise_params['mz_range'][1]
spike_noise = UniformSpikeNoise(noise_density, noise_max_val, min_mz=noise_min_mz,
                                max_mz=noise_max_mz)
mass_spec = IndependentMassSpectrometer(ionisation_mode, chems, spike_noise=spike_noise)

In [23]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, out_file='TopN_controller.mzML')
env.run()

  0%|          | 0/800 [00:00<?, ?it/s]

In [24]:
evaluate(env)

{'coverage_prop': '0.755',
 'intensity_prop': '0.643',
 'ms1/ms2 ratio': '0.104',
 'efficiency': '0.556'}

### Test on QCB data

#### Load pre-processed QCB chemicals

In [40]:
fullscan_file = 'fullscan_QCB.mzML'

In [41]:
# min_roi_intensity = 0
# min_roi_length = 0

min_roi_intensity = 500
min_roi_length = 0
at_least_one_point_above = 5000

In [46]:
filename = 'datasets_%d_%d_%d.p' % (min_roi_intensity, min_roi_length, at_least_one_point_above)

if exists(filename):
    chemicals = load_obj(filename)
    print(len(chemicals))
else:
    rp = RoiBuilderParams(min_roi_intensity=min_roi_intensity, min_roi_length=min_roi_length, 
                   at_least_one_point_above=at_least_one_point_above)
    cm = ChemicalMixtureFromMZML(fullscan_file, roi_params=rp)
    chemicals = cm.sample(None, 2, source_polarity=ionisation_mode)
    print(len(chemicals))
    save_obj(chemicals, filename)

19889


#### Filter chemicals by mz and RT range

In [47]:
filtered = []
for chem in chemicals:
    if (min_mz < chem.isotopes[0][0] < max_mz) and (min_rt < chem.rt < max_rt):
        filtered.append(chem)
        
len(filtered)

16544

In [48]:
filtered_chem_list = [filtered]

#### Disable spike noise in DDAEnv

In [49]:
copy_params = dict(params)
copy_params['noise']['enable_spike_noise'] = False
copy_params

{'chemical_creator': {'mz_range': (100, 600),
  'rt_range': (200, 1000),
  'intensity_range': (10000.0, 10000000000.0),
  'n_chemicals': (2000, 5000),
  'mz_sampler': <vimms.ChemicalSamplers.MZMLFormulaSampler at 0x7fa8665a6b80>,
  'ri_sampler': <vimms.ChemicalSamplers.MZMLRTandIntensitySampler at 0x7fa8665a61c0>,
  'cr_sampler': <vimms.ChemicalSamplers.MZMLChromatogramSampler at 0x7fa8665a67c0>},
 'noise': {'enable_spike_noise': False,
  'noise_density': 0.1,
  'noise_max_val': 1000.0,
  'mz_range': (100, 600)},
 'env': {'ionisation_mode': 'Positive',
  'rt_range': (200, 1000),
  'isolation_window': 0.7,
  'mz_tol': 10,
  'rt_tol': 15}}

#### Test different methods

In [50]:
for method in methods:
    banner = 'method = %s max_peaks = %d rt_tol = %d' % (method, max_peaks, rt_tol)
    print(banner)
    print()
    
    if method == 'PPO':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = PPO.load(fname)
    elif method == 'DQN':
        fname = os.path.join(in_dir, 'model_%s.zip' % method)
        model = DQN.load(fname)
    else:
        model = None

    env = DDAEnv(max_peaks, copy_params)
    run_method(env, filtered_chem_list, method, out_dir, min_ms1_intensity=min_ms1_intensity, model=model, 
               N=10, mzml_prefix='QCB')
    print()

method = PPO max_peaks = 100 rt_tol = 15

Episode 0 finished after 3201 timesteps with reward 255.28604649136412

method = TopN max_peaks = 100 rt_tol = 15

Episode 0 finished after 3569 timesteps with reward 543.634210174385

method = random max_peaks = 100 rt_tol = 15

Episode 0 finished after 3958 timesteps with reward 436.728098486703



#### Test classic Top-N controller in ViMMS

In [51]:
mass_spec = IndependentMassSpectrometer(ionisation_mode, filtered)

In [52]:
controller = TopNController(ionisation_mode, N, isolation_window, mz_tol, rt_tol, min_ms1_intensity)
env = Environment(mass_spec, controller, min_rt, max_rt, progress_bar=True, out_dir=out_dir, 
                  out_file='QCB_TopN_controller.mzML')
env.run()

  0%|          | 0/800 [00:00<?, ?it/s]

#### Evaluation from mzML

In [53]:
boxes = load_picked_boxes('fullscan_QCB_box.csv')

In [54]:
mzmls = [
    os.path.join(out_dir, 'QCB_PPO_0.mzML'),
    os.path.join(out_dir, 'QCB_TopN_0.mzML'),
    os.path.join(out_dir, 'QCB_TopN_controller.mzML'),
    os.path.join(out_dir, 'QCB_random_0.mzML'),    
]
for mzml in mzmls:
    res = evaluate_real([mzml], boxes)
    cov_prop = res['coverage_prop']
    int_prop = res['intensity_prop']
    print(mzml, cov_prop, int_prop)

evaluation/eval_100_15/QCB_PPO_0.mzML [0.0952048417132216] [0.10614815755248841]
evaluation/eval_100_15/QCB_TopN_0.mzML [0.21042830540037244] [0.2518706853059417]
evaluation/eval_100_15/QCB_TopN_controller.mzML [0.22486033519553073] [0.258920200372677]
evaluation/eval_100_15/QCB_random_0.mzML [0.12057728119180633] [0.28771245625707886]
